In [86]:
import pandas as pd
import numpy as np
import glob
import json
import os
import random
import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_model
from botorch.utils import standardize
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.acquisition import UpperConfidenceBound, ExpectedImprovement, NoisyExpectedImprovement
from botorch.optim import optimize_acqf

seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

def load_data(user_id, session_id, task_id):
    files = glob.glob(f'{user_id}-{session_id}/feedbacks/*.json')
    df = pd.DataFrame(columns=[
        'userID',
        'studyID',
        'sessionID',
        'taskID',
        'feedbackID',
        'iteration',
        'completeIteration',
        'incompleteIteration',
        'rating',
        'skip',
        'timePresent',
        'timeLastRating',
        'comment'
    ])
    params = ['reduction_ratio', 'creativity', 'repetition_penalty', 'frequency_penalty']
    for feedback_file in files:
        param_file = feedback_file.replace('feedbacks', 'parameters')
        feedback = json.load(open(feedback_file))
        param    = json.load(open(param_file))
        feedback.update(param)
        df = pd.concat([df, pd.DataFrame([feedback])], ignore_index=True)
    df = df[(df.skip == False) & (df.taskID == task_id)]
    X = torch.tensor(df[params].to_numpy(dtype=np.float64))
    y = torch.tensor(df.rating.to_numpy(dtype=np.float64)).unsqueeze(0).T
    return X, y, df

def bayesian_optimization(X, y, dim):
    gp = SingleTaskGP(X, y)
    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_model(mll)
    bounds = torch.stack([torch.zeros(dim), torch.ones(dim)])
    candidate, acq_value = optimize_acqf(
        NoisyExpectedImprovement(gp), bounds=bounds, q=1, num_restarts=5, raw_samples=20,
    )
    return candidate, acq_value


In [90]:
X, y, df = load_data('changkun', 'testsession', '1')
candidate, acq_value = bayesian_optimization(X, y)
print(candidate, acq_value)

tensor([[0.2680, 0.9195, 0.7581, 0.8376]]) tensor(71.0518, dtype=torch.float64)


In [ ]:
class Participant:
    def __init__(self, user_id, session_id):
        self.user_id = user_id
        self.session_id = session_id
        self.df_begin, self.df_iter, self.df_end = self.__load_participant(user_id, session_id)

    def __load_participant(self, user_id, session_id):
        files = glob.glob(f'{user_id}-{session_id}/feedbacks/*.json')
        df_iter = pd.DataFrame(columns=[
            'userID',
            'studyID',
            'sessionID',
            'taskID',
            'feedbackID',
            'iteration',
            'completeIteration',
            'incompleteIteration',
            'rating',
            'skip',
            'timePresent',
            'timeLastRating',
            'comment'
        ])
        for feedback_file in files:
            param_file = feedback_file.replace('feedbacks', 'parameters')
            feedback = json.load(open(feedback_file))
            param    = json.load(open(param_file))
            feedback.update(param)
            df_iter = pd.concat([df_iter, pd.DataFrame([feedback])], ignore_index=True)
        df_iter.sort_values(by=['timePresent'], inplace=True)
        df_iter.reset_index(inplace=True, drop=True)

        scales = glob.glob(f'{user_id}-{session_id}/*-finish.json')
        df_end = pd.DataFrame(columns=[
            'confidence',
            'confidenceTimestamp',
            'expectation',
            'expectationTimestamp',
            'finishTimestamp',
            'regret',
            'regretTimestamp',
            'satisfaction',
            'satisfactionTimestamp',
            'sessionID',
            'studyID',
            'taskID',
            'userID',
        ])
        for scale_file in scales:
            scale = json.load(open(scale_file))
            df_end = pd.concat([df_end, pd.DataFrame([scale])], ignore_index=True)
        df_end.sort_values(by=['confidenceTimestamp'], inplace=True)
        df_end.reset_index(inplace=True, drop=True)

        begin = glob.glob(f'{user_id}-{session_id}/*-begin.json')[0]
        begin = json.load(open(begin))
        df_begin = pd.DataFrame([begin])

        return df_begin, df_iter, df_end

p = Participant('P038', 'XS0435')
p.df_iter.sort_values(by=['timePresent'], inplace=True)
p.df_iter.reset_index(inplace=True, drop=True)
p.df_iter.rating.plot()
p.df_end